# Reproducing Asari & Zador (2009) from CRCNS ac-1

**Paper.** Asari H, Zador AM (2009). Long-lasting context dependence constrains neural encoding
models in rodent auditory cortex. *J Neurophysiol* 102:2638-2656.
[doi:10.1152/jn.00577.2009](https://doi.org/10.1152/jn.00577.2009)

**Dataset.** CRCNS ac-1, Asari, Wehr, Machens & Zador (2009), "Auditory cortex and thalamic
neuronal responses to various natural and synthetic sounds",
[doi:10.6080/K0KW5CXR](https://doi.org/10.6080/K0KW5CXR). Streamed from the public S3 bucket
`crcnsarchive` (`ac-1/`), zip files opened in place.

**Version.** CRCNS data sets are not versioned, so the version is recorded as `unversioned`
(docs/archives.md, CRCNS). The paper cites no version. The files read and their MD5 sums from
the data set's `checksums.md5` (2026-11-14) are `crcns-ac1.zip` 2300ee76d93383ab8c808ddf16761a23,
`crcns-ac1-asari-results-1.zip` 18537da81d98e311c0274f62aea04daa and
`crcns-ac1-asari-results-2.zip` 2f7ab955cc778fc6839ce59ed51bce89; a later change of the files
would change these sums. This is the only version the archive offers, and its contents match the
claims file's description (39 natural-sound A1 cells, 14 MGB cells, the synthetic-sound cells).

**Claims.** `claims/asari-2009.yaml`, written from the version of record (text extraction of the
publisher's PDF); the supplement was not read by the claims reader.

**Code.** The paper links no code. The data set ships the authors' analysis scripts
(`asari/Tools/CxtDep.m`, "Basic analysis script for Asari and Zador (2008)", with
`ViewResp.m`, `GetEvoked.m`, `GetSpikes.m`; `wehr/Tools/STRFanalysis.m`). They are followed here
where the Methods leave a detail open; `asari.py` says which detail comes from where.

| claim | tested here |
|---|---|
| methods-recording-counts | tested (voltage-clamp counts: no voltage-clamp recordings in the data set) |
| a1-mgb-firing-rates | tested |
| fig2-example-reliability | tested |
| fig2-example-long-context | tested |
| fig3a-proportion-significant | tested |
| fig3a-significance-decay | tested |
| fig3b-fractional-power-decay | tested |
| timescale-longer-than-membrane | tested |
| fig4-predictability-upper-bound | tested |
| fig4-strf-performance-low | see its section |
| fig5-example-conductances | blocked: no voltage-clamp recordings |
| fig6-conductance-decay | blocked: no voltage-clamp recordings |
| fig8a-proportion-by-property | tested |
| fig8b-decay-by-property | tested |
| fig8a-significance-fits | tested |
| fig8c-relative-effects | tested |
| fig8-natural-synthetic-probes-alike | tested |
| fig9-mgb-fast-decay | tested |
| fig9-example-first-bin | tested |
| fig7-example-frequency-context | blocked: no recording with the Figure 7 design |
| figs1-silence-probe | tested |

In [ ]:
import time
import resource
import collections
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import asari

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
T0 = time.time()
A = asari.Archive()
rows = pd.DataFrame(asari.summary(A))
print("zip members:", len(A.members))

## Loading

Every recording the analyses use is read once and reduced as it loads to the per-probe measures
(significance at 0.1-ms resolution, stored as 1-ms means, and the power terms). Seeds: each file
has its own generator for the shuffles of the noise floor (`seed` below), so that adding a file
does not change the numbers of another.

In [ ]:
SEED_NAT, SEED_NAT_NOSIL, SEED_SYN = 10_000, 20_000, 30_000
nat_rows = rows[(rows.experiment == 0) & (rows.description != "tuning curve")].reset_index(drop=True)
mgb_rows = rows[(rows.experiment == 1) & (rows.description != "tuning curve")].reset_index(drop=True)
syn_rows = rows[(rows.experiment == 2) & (rows.description != "tuning curve")].reset_index(drop=True)
import os
if os.environ.get("ASARI_DEV_SUBSET"):   # development only: a few files of each group
    k = int(os.environ["ASARI_DEV_SUBSET"])
    nat_rows, mgb_rows, syn_rows = nat_rows.iloc[15:15 + k], mgb_rows.iloc[:k], syn_rows.iloc[::max(1, len(syn_rows) // k)]

t = time.time()
nat_cells, nat_probes = [], []
for i, r in nat_rows.iterrows():
    c, res = asari.analyze_cc_cell(A, A.result_member(r.file), seed=SEED_NAT + i, silence_probe=True)
    c.update(cell=r.cell, description=r.description)
    nat_cells.append(c)
    for x in res:
        x["cell"] = r.cell
    nat_probes += res
print(f"A1 natural sounds: {len(nat_rows)} files in {time.time() - t:.0f} s")

# reading without "silence" among the contexts (claims fig3a-proportion-significant, fig3b, fig4)
t = time.time()
nat_probes_nosil = []
for i, r in nat_rows.iterrows():
    c, res = asari.analyze_cc_cell(A, A.result_member(r.file), seed=SEED_NAT_NOSIL + i,
                                   include_silence=False, extras=False)
    for x in res:
        x["cell"] = r.cell
    nat_probes_nosil += res
print(f"A1 natural sounds, silence left out: {time.time() - t:.0f} s")

t = time.time()
mgb = {}
for det in ["getspikes", "mad:5", "mad:8"]:
    infos, probes = [], []
    for i, r in mgb_rows.iterrows():
        info, out = asari.analyze_mgb_file(A, A.result_member(r.file), det)
        info["cell"] = r.cell
        infos.append(info)
        for x in out:
            x["cell"] = r.cell
        probes += out
    mgb[det] = (infos, probes)
print(f"MGB: {len(mgb_rows)} files x 3 spike detectors in {time.time() - t:.0f} s")

t = time.time()
syn_infos, syn_entries = [], []
for i, r in syn_rows.iterrows():
    info, out = asari.analyze_synthetic_file(A, A.result_member(r.file), seed=SEED_SYN + i)
    info.update(cell=r.cell, description=r.description)
    syn_infos.append(info)
    for x in out:
        x["cell"] = r.cell
    syn_entries += out
print(f"A1 synthetic sounds: {len(syn_rows)} files in {time.time() - t:.0f} s")
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB; "
      f"elapsed {(time.time() - T0) / 60:.1f} min")

## Data check

In [ ]:
print("Summary.mat experiments (cells, files other than tuning curves):")
for e, g in rows.groupby("experiment"):
    print(f"  {e}: {g.experiment_description.iloc[0]!r} | reference {g.reference.iloc[0][:40]!r}: "
          f"{g.cell.nunique()} cells, {(g.description != 'tuning curve').sum()} files")
listed = set(rows.file.str.replace("Results/", "", regex=False))
in_zip = {m.split("/", 1)[1] for m in A.members if m.startswith("asari-results-") and m.endswith(".mat")}
print("files listed in Summary.mat but not in the zip files:", sorted(listed - in_zip))
print("files in the zip files not listed in Summary.mat:", sorted(in_zip - listed))

In [ ]:
def recinfo(infos, kind):
    df = pd.DataFrame([{"kind": kind, "cell": i["cell"], "type": i["recording_type"],
                        "sf": i["checks"]["sf"], "rep": i["rep"],
                        "pres_min": i["checks"]["presentations_min"], "pres_max": i["checks"]["presentations_max"],
                        "dur_mismatch_ms": i["checks"]["duration_mismatch_ms"],
                        "gap_min_s": i["checks"]["gap_min"], "gap_median_s": i["checks"]["gap_median"],
                        "gap_max_s": i["checks"]["gap_max"],
                        "onset_effect": i["checks"]["onset_effect"][0], "onset_p": i["checks"]["onset_effect"][1],
                        "shifted_effect": i["checks"]["shifted_effect"][0], "shifted_p": i["checks"]["shifted_effect"][1]}
                       for i in infos])
    return df

chk = pd.concat([recinfo(nat_cells, "A1 natural"), recinfo(mgb["getspikes"][0], "MGB"),
                 recinfo(syn_infos, "A1 synthetic")])
print("recording types by group:")
print(chk.groupby(["kind", "type"]).size().to_string())
print("\nsampling rates:", sorted(chk.sf.unique()))
print("\nby group: files, effective repeats (range), presentations per sequence (range), largest mismatch"
      " between the sum of fragment durations and the trigger duration (ms), gaps between sequences (s)")
print(chk.groupby("kind").agg(files=("cell", "size"), cells=("cell", "nunique"), rep_min=("rep", "min"),
                              rep_max=("rep", "max"), pres_min=("pres_min", "min"), pres_max=("pres_max", "max"),
                              mismatch_max=("dur_mismatch_ms", "max"), gap_min=("gap_min_s", "min"),
                              gap_median=("gap_median_s", "median"), gap_max=("gap_max_s", "max")).round(3).to_string())
print("\nfiles with fewer presentations of a sequence than the stated effective repeats:",
      int((chk.pres_min < chk.rep).sum()), "of", len(chk))
print(chk[chk.pres_min < chk.rep][["kind", "cell", "rep", "pres_min", "pres_max"]].to_string())
mis = pd.DataFrame([{"kind": k, "file": i["member"].split("/")[-1], "stored_mismatch_ms": i["checks"]["stored_mismatch_ms"],
                     "fragment list taken from": i["checks"]["stimulus_source"].split("/")[-1],
                     "mismatch_after_ms": i["checks"]["duration_mismatch_ms"]}
                    for k, infos in [("A1 natural", nat_cells), ("MGB", mgb["getspikes"][0]), ("A1 synthetic", syn_infos)]
                    for i in infos if i["checks"]["stored_mismatch_ms"] > asari.MAX_MISMATCH_MS])
print("\nfiles whose stored fragment durations do not add up to the trigger durations (> 5 ms):")
print(mis.round(1).to_string())

In [ ]:
# Clock check: response in the 0.3 s after each sequence onset minus the 0.3 s before it, against
# the same contrast with the onsets moved 3 s back into the silence (membrane potential in mV for
# A1, spike rate in Hz for MGB). Wilcoxon signed-rank over sequences, per file.
g = chk.groupby("kind")
print(pd.DataFrame({
    "files": g.size(),
    "onset effect median": g.onset_effect.median(),
    "files p<0.01 at onset": g.onset_p.apply(lambda p: int((p < 0.01).sum())),
    "shifted effect median": g.shifted_effect.median(),
    "files p<0.01 shifted": g.shifted_p.apply(lambda p: int((p < 0.01).sum())),
}).round(3).to_string())
print("\nfiles without an onset effect at p<0.01:")
print(chk[chk.onset_p >= 0.01][["kind", "cell", "onset_effect", "onset_p"]].round(4).to_string())

In [ ]:
tr = pd.DataFrame([{"kind": "A1 natural", "min": c["trace_min"], "max": c["trace_max"], "unit": c["unit"]} for c in nat_cells]
                  + [{"kind": "MGB", "min": i["trace_min"], "max": i["trace_max"], "unit": i["unit"]} for i in mgb["getspikes"][0]])
print("trace ranges after the 10-ms median filter (A1) or raw (MGB):")
print(tr.groupby(["kind", "unit"]).agg(min_lo=("min", "min"), min_hi=("min", "max"), max_lo=("max", "min"),
                                       max_hi=("max", "max")).round(1).to_string())
el_nat = [p for p in nat_probes if p["eligible"]]
print("\nA1 natural probes: durations (s)", np.round(np.percentile([p["duration"] for p in el_nat], [0, 50, 100]), 2),
      "| contexts per probe", dict(collections.Counter(p["n_contexts"] for p in el_nat)),
      "| trials per context", dict(sorted(collections.Counter(p["rep"] for p in el_nat).items())))

Checks of the description against the data are stated with the numbers above: all A1 recordings
are whole-cell (current clamp) and all MGB recordings cell-attached, at 10 kHz, in mV. No
recording in the data set is a voltage-clamp recording. The fragment durations stored with each
recording add up to the trigger duration of each sequence to within a fraction of a millisecond,
except in the files listed: in two cells (one A1, one MGB) the fragment lists of a cell's two
files are swapped, the stored durations of each file fitting the triggers of the other exactly,
and here each file's probes are cut with the list that fits its triggers; in one synthetic
recording the triggers are 308 ms longer than the seven stored fragment durations (4200 ms each)
in every sequence; no other file fits, and the stored durations are used, as the authors'
`CxtDep.m` would use them. The gaps between sequences are about 6 s for the natural-sound recordings, as the
Methods say, and shorter in others (minimum and median above). Every sequence onset evokes a
response in nearly every file while the same contrast moved into the silence does not, which is
consistent with the triggers sharing the clock of the traces; files without an onset effect are
listed.

## methods-recording-counts

> Of 194 A1 cells recorded in current clamp, 123 met the analysis criterion (39 tested with
> natural sounds, 14 with TORCs, 6 with DMRs, 39 with modulated harmonic tones, 27 with modulated
> colored noise); 14 of 42 voltage-clamped cells and 14 of 16 MGB cells were analysed.

In [ ]:
def counts_cc(probes):
    el = [p for p in probes if p["eligible"]]
    return len(el), len({p["cell"] for p in el})

n_p, n_c = counts_cc(nat_probes)
n_p2, n_c2 = counts_cc(nat_probes_nosil)
print(f"A1 natural-sound probes (silence counted as a context): {n_p} probes in {n_c} cells "
      f"(of {len(nat_probes)} probes with at least two contexts)")
print(f"A1 natural-sound probes (silence not counted):          {n_p2} probes in {n_c2} cells")
mgb_el = [p for p in mgb["getspikes"][1] if p["eligible"]]
print(f"MGB probes (at least 10 trials per context): {len(mgb_el)} probes in {len({p['cell'] for p in mgb_el})} cells "
      f"(Summary.mat lists {mgb_rows.cell.nunique()} MGB cells)")
syn_cells_any = {i["cell"] for i in syn_infos if i["n_eligible_any"] > 0}
syn_cells_prop = {e["cell"] for e in syn_entries if e["eligible"]}
nat_cells_set = {p["cell"] for p in nat_probes if p["eligible"]}
print(f"A1 synthetic-sound cells with an eligible probe (any contexts): {len(syn_cells_any)}; "
      f"with an eligible probe x property ensemble: {len(syn_cells_prop)} (Summary.mat lists {syn_rows.cell.nunique()})")
print(f"A1 current-clamp cells meeting the criterion, natural + synthetic: {len(nat_cells_set | syn_cells_any)} "
      f"(any contexts) or {len(nat_cells_set | syn_cells_prop)} (property ensembles); overlap {len(nat_cells_set & syn_cells_any)}")
fam = {i["member"]: i["trigger_type"] for i in syn_infos}
fam_cells = collections.defaultdict(set)
for e in syn_entries:
    if e["eligible"]:
        fam_cells[fam[e["member"]]].add(e["cell"])
print("synthetic cells by stimulus family (cells with an eligible ensemble):",
      {k: len(v) for k, v in sorted(fam_cells.items())}, "| paper: TORC 14, DMR 6, MHT ('synthetic') 39, MCN ('pinknoise') 27")
print("recording types in the whole data set:", dict(chk.groupby("type").size()), "| voltage-clamp recordings: 0")

**Result: partially reproduced.** The natural-sound A1 counts match exactly: 305 probes in 39 cells, under both readings of silence. The MGB counts match exactly: 93 probes in 14 cells. The A1 current-clamp total is 132 cells (39 natural, 95 synthetic, 2 in both) against 123, and the cells per stimulus family do not map onto the paper's (TORC 14 matches; the other families differ). The voltage-clamp counts (29 probes, 14 cells) cannot be tested: the data set holds no voltage-clamp recording.

## a1-mgb-firing-rates

> Firing rates were low in A1 (spontaneous 0.47 +/- 0.61 Hz, evoked 0.57 +/- 0.77 Hz, mean +/- SD,
> 194 cells) and high in MGB (spontaneous 0.78 +/- 1.25 Hz, evoked 11.4 +/- 16.9 Hz, 16 cells).

Readings of the windows (criterion): spontaneous is the whole intersequence silence (`spont`) or
the last 1 s before each sequence (`pre1s`); evoked is all sound periods (the analysed probes
cover all fragments of the natural-sound sequences, so the two evoked readings coincide). MGB
spikes: the authors' automatic detector (`GetSpikes.m`) is the main detector; robust thresholds
of 5 and 8 SD on the same filter are variants. A1 spikes: upward crossings of -20 mV on the raw
membrane potential (assumption; -10 and -30 mV shown). Rates are per cell, pooling a cell's
files, then averaged over cells.

In [ ]:
def cell_rates(infos, key=None):
    agg = {}
    for i in infos:
        r = i["rates"][key] if key is not None else i["rates"]
        a = agg.setdefault(i["cell"], dict(sp=0.0, sp_t=0.0, ev=0.0, ev_t=0.0, pre=[]))
        a["sp"] += r["spont"] * r["spont_s"]; a["sp_t"] += r["spont_s"]
        a["ev"] += r["evoked"] * r["evoked_s"]; a["ev_t"] += r["evoked_s"]
        a["pre"].append(r["pre1s"])
    return pd.DataFrame([{"cell": c, "spont": a["sp"] / a["sp_t"], "evoked": a["ev"] / a["ev_t"],
                          "pre1s": np.nanmean(a["pre"])} for c, a in agg.items()])

rate_rows = []
a1_infos = nat_cells + syn_infos
for thr in asari.A1_THRESHOLDS:
    d = cell_rates(a1_infos, thr)
    for w in ["spont", "pre1s", "evoked"]:
        rate_rows.append(dict(area="A1", detector=f"Vm {thr:+.0f} mV", window=w, cells=len(d),
                              mean=d[w].mean(), sd=d[w].std(ddof=1)))
for det, (infos, _) in mgb.items():
    d = cell_rates(infos)
    for w in ["spont", "pre1s", "evoked"]:
        rate_rows.append(dict(area="MGB", detector=det, window=w, cells=len(d), mean=d[w].mean(), sd=d[w].std(ddof=1)))
rates = pd.DataFrame(rate_rows)
exp = {("A1", "spont"): 0.47, ("A1", "pre1s"): 0.47, ("A1", "evoked"): 0.57,
       ("MGB", "spont"): 0.78, ("MGB", "pre1s"): 0.78, ("MGB", "evoked"): 11.4}
rates["paper"] = [exp[(a, w)] for a, w in zip(rates.area, rates.window)]
rates["within_10pct"] = (rates["mean"] - rates.paper).abs() <= 0.1 * rates.paper
print(rates.round(3).to_string())

**Result: not reproduced (resting on the choice of spike detector).** With the detectors adopted (A1: -20 mV on the raw membrane potential; MGB: the authors' automatic detector), A1 rates are 0.392 Hz spontaneous and 0.439 Hz evoked over 133 cells (paper 0.47 and 0.57), and MGB rates 0.549 and 4.99 Hz over 14 cells (paper 0.78 and 11.4). No reading of the windows brings all four means within 10%. The partial rule fails narrowly: A1 means are below 1 Hz but the MGB evoked mean is 4.99 Hz, under 5 Hz. With a robust 8-SD threshold on the same filter (a variant) the MGB evoked mean is 10.38 Hz, within tolerance, and the spontaneous mean 0.975 Hz is not.

## fig2-example-reliability

> The example A1 neuron of Figure 2 had high trial-to-trial reliability (correlation across trials
> 0.74 +/- 0.08 for the shown probe and 0.61 +/- 0.07 over the seven natural sound fragments
> tested), and the mean correlation over the population was 0.31 +/- 0.09.

Readings (criterion): the coefficient as the mean over pairs of trials (`pairs`) or each trial
against the mean of the others (`loo`), within each context over the probe window, averaged over
contexts; the population mean over cells or over probes.

In [ ]:
cp = pd.DataFrame([{"cell": p["cell"], "member": p["member"], "probe": p["probe"], "pairs": p["corr"]["pairs"],
                    "loo": p["corr"]["loo"]} for p in el_nat])
per_cell = cp.groupby("cell")[["pairs", "loo"]].mean()
print("population, over cells (n = %d):" % len(per_cell))
print(per_cell.agg(["mean", "std"]).round(3).to_string())
print("population, over probes (n = %d):" % len(cp))
print(cp[["pairs", "loo"]].agg(["mean", "std"]).round(3).to_string())
# candidate example cells: seven natural fragments, six repeats (Figure 2A)
cand = [c for c in nat_cells if c["n_fragments"] == 7 and c["rep"] == 6]
rowsx = []
for c in cand:
    sub = cp[cp.member == c["member"]]
    rowsx.append(dict(cell=c["cell"], probes=len(sub), cell_pairs_mean=sub.pairs.mean(), cell_pairs_sd=sub.pairs.std(ddof=1),
                      cell_loo_mean=sub.loo.mean(), best_probe_pairs=sub.pairs.max(),
                      best_probe=int(sub.probe[sub.pairs.idxmax()])))
print("\ncandidate example cells (7 fragments, 6 repeats):")
print(pd.DataFrame(rowsx).round(3).to_string())

**Result: reproduced.** The mean pairwise trial-to-trial correlation over the 39 cells is 0.310 +/- 0.092 (paper 0.31 +/- 0.09); over the 305 probes it is 0.319. One candidate example cell (20051222-ha1-001, seven fragments, six repeats) has a mean of 0.611 +/- 0.081 over its probes (paper 0.61 +/- 0.07) and 0.737 for probe 4 (paper 0.74), the probe heard after silence and two other fragments. The leave-one-out reading gives higher values (0.437 over cells).

## fig2-example-long-context

> In an example A1 neuron, changing the conditioning stimulus changed the response to the probe
> for about 4 s; responses did not differ between 2 and 4 s after probe onset and diverged again
> after about 4 s.

In [ ]:
def pattern(runs):
    return dict(before_2s=any(a < 2 for a, b in runs), in_2_4s=any(b > 2 and a < 4 for a, b in runs),
                after_4s=any(b > 4 for a, b in runs))

rows2 = []
for c in [c for c in nat_cells if c["n_fragments"] == 7]:
    for p in el_nat:
        if p["member"] == c["member"] and p["n_contexts"] == 3 and 0 in p["contexts"] and p["rep"] >= 6:
            rows2.append(dict(cell=p["cell"], probe=p["probe"], dur=round(p["duration"], 2), contexts=p["contexts"],
                              trials=p["rep"], pairs=round(p["corr"]["pairs"], 3), **pattern(p["runs"]),
                              runs_after_1_5s=[(round(a, 3), round(b, 3)) for a, b in p["runs"] if b > 1.5]))
ex = pd.DataFrame(rows2)
print(f"probes with silence and two fragments as contexts, at least 6 trials, in 7-fragment cells: {len(ex)}")
print(ex.to_string())
print("\nprobes with significant periods before 2 s and after 4 s and none in 2-4 s:",
      int((ex.before_2s & ex.after_4s & ~ex.in_2_4s).sum()))
print("natural-sound probes with a significant period later than 4 s:",
      sum(any(b > 4 for a, b in p["runs"]) for p in el_nat), "of", len(el_nat))

**Result: reproduced.** In the cell matched by its reliability (20051222-ha1-001, probe 4, contexts silence and fragments 2 and 3, six trials each), significant periods occur up to 1.60 s, none from 1.60 to 3.97 s, and again from 3.97 to 4.60 s. The one period that touches the 2 to 4 s window starts at 3.970 s, the paper's "after about 4 s" (within 10% of 4 s); read strictly as "none between 2 and 4 s", it overlaps the window by 30 ms. Two other probes of this design meet the strict pattern (20060105-ha1-001 probe 4, 20060225-ha1-001 probe 5). Over all natural-sound probes, 15 of 305 have a significant period later than 4 s.

## fig3a-proportion-significant

> Of 305 natural sound probes tested in 39 A1 cells with typically five to eight natural sound
> contexts, 204 (66.9%) showed significant context dependence, and about a quarter (23.7%) of the
> significant events occurred more than 1 s after probe onset.

Readings (criterion): "silence" counted among the contexts or not; the proportion at time t over
all 305 probes (ended probes count as not significant) or over the probes still running at t.

In [ ]:
def prop_curves(probes, reading):
    el = [p for p in probes if p["eligible"]]
    prop = asari.population_mean(asari.stack(el, "signif"), reading)
    noise = asari.population_mean(asari.stack(el, "noise"), reading)
    return el, prop, noise

out3 = []
curves3 = {}
for sil, probes in [("silence counted", nat_probes), ("silence left out", nat_probes_nosil)]:
    for reading in ["all", "running"]:
        el, prop, noise = prop_curves(probes, reading)
        lf, tpk, cum = asari.late_fraction(prop, noise)
        nsig = sum(p["significant"] for p in el)
        curves3[(sil, reading)] = (prop, noise)
        out3.append(dict(contexts=sil, proportion_over=reading, probes=len(el), significant=nsig,
                         fraction=nsig / len(el), late_fraction=lf, cum_peak_s=tpk,
                         mean_noise_0_6s=float(np.nanmean(noise[:6000]))))
out3 = pd.DataFrame(out3)
out3["count_ok"] = (out3.significant - 204).abs() <= 20.4
out3["fraction_ok"] = (out3.fraction - 0.669).abs() <= 0.0669
out3["late_ok"] = (out3.late_fraction - 0.237).abs() <= 0.0237
print(out3.round(4).to_string())
nsig_cell = collections.Counter(p["cell"] for p in el_nat if p["significant"])
print("\ncells with at least one significant probe:", len(nsig_cell), "of", len({p['cell'] for p in el_nat}))

prop, noise = curves3[("silence counted", "all")]
tt = np.arange(len(prop)) / 1000
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(tt, prop, "k", lw=1, label="proportion significant")
ax.plot(tt, noise, color="0.6", lw=1, label="noise floor")
ax.set_xlim(0, 6.1); ax.set_xlabel("time from probe onset (s)"); ax.set_ylabel("probability"); ax.legend()
print(f"plotted range: proportion {np.nanmin(prop):.4f} to {np.nanmax(prop):.4f}, noise {np.nanmin(noise):.5f} to {np.nanmax(noise):.5f}, all points within the axes")

**Result: partially reproduced.** 183 of 305 probes (60.0%) show significant context dependence, against 204 (66.9%): the count is 21 below the paper's (tolerance 20.4) and the fraction 0.002 below its lower bound. The late fraction is within tolerance: 0.225 over all probes and 0.237 over running probes (paper 0.237), with the cumulative curve peaking at 4.718 s (paper 4.72 s). Leaving silence out of the contexts gives 160 significant probes and a late fraction of 0.139 to 0.144. The criterion's partial rule names the opposite pattern (count within, late fraction not); the outcome here is a weaker effect with the timing reproduced.

## fig3a-significance-decay

> The proportion of probes with significant context dependence decays after probe onset as the
> sum of two exponentials (sizes 0.17 and 0.09, decay constants 0.20 and 0.90 s), above a noise
> floor of about 2.8 x 10^-3.

Readings: fit range 0 to 4.72 s or 0 to 6 s (criterion); bounds on the sizes (none, or sizes of
a decay kept non-negative; the paper gives no bounds); the proportion readings of the claim it
depends on. The constant is fixed at the mean noise floor over 0 to 6 s of the same reading.

In [ ]:
fit3 = []
for (sil, reading), (prop, noise) in curves3.items():
    tt = np.arange(len(prop)) / 1000
    kap = float(np.nanmean(noise[:6000]))
    for hi in [4.72, 6.0]:
        m = tt <= hi
        for nonneg in [False, True]:
            n, fits = asari.choose_n_exp(tt[m], prop[m], kappa=kap, nonneg=nonneg)
            f2 = fits[2]["params"]
            fit3.append(dict(contexts=sil, over=reading, range=hi, nonneg_sizes=nonneg, chosen=n,
                             a1=f2[0][0], tau1=f2[0][1], a2=f2[1][0], tau2=f2[1][1], kappa=kap,
                             one_exp=(round(fits[1]["params"][0][0], 3), round(fits[1]["params"][0][1], 3))))
fit3 = pd.DataFrame(fit3)
tol = dict(a1=(0.17, 0.017), tau1=(0.20, 0.02), a2=(0.09, 0.009), tau2=(0.90, 0.09), kappa=(0.0028, 0.00028))
for k, (v, d) in tol.items():
    fit3[k + "_ok"] = (fit3[k] - v).abs() <= d
fit3["all_ok"] = fit3[[k + "_ok" for k in tol]].all(1) & (fit3.chosen == 2)
fit3["partial"] = (fit3.chosen == 2) & fit3.tau2.between(0.5, 2.0)
print(fit3.round(4).to_string())

**Result: partially reproduced (resting on non-negative sizes).** With decay sizes kept non-negative, two exponentials are selected in every reading with silence counted, with a slow constant of 0.90 to 0.97 s (paper 0.90 s) and the fitted constant at the mean noise floor of 0.0028 for running probes (paper 2.8 x 10^-3); the fast constant (0.25 to 0.26 s against 0.20 s) and the slow size (0.070 to 0.080 against 0.09) are outside tolerance, and the fast size is within it in one reading (0.154). Without bounds the fit takes a negative fast component (-0.17, 0.015 s) for the rise in the first 50 ms, which is not a decay.

## fig3b-fractional-power-decay

> The context-dependent fraction of the stimulus-related response power, normalized by the
> smoothed stimulus-related power, decays from about 0.49 with a time constant of about 1.04 s.

Q(t) = (Eq. 17 - Eq. 18) / running mean of Eq. 17 over [0, 2t]; one exponential with constant 0.
Readings: silence counted or not; probes averaged over those running at t or all (ended probes
as 0); fit range 0 to 4 s or 0 to 6 s.

In [ ]:
fitq, qcurves = [], {}
for sil, probes in [("silence counted", nat_probes), ("silence left out", nat_probes_nosil)]:
    el = [p for p in probes if p["eligible"]]
    for reading in ["running", "all"]:
        q = asari.q_curves(el, reading)
        qcurves[(sil, reading)] = q
        for hi in [4.0, 6.0]:
            m = (q["t"] <= hi) & np.isfinite(q["Q"])
            f = asari.fit_exponentials(q["t"][m], q["Q"][m], 1, kappa=0.0)
            g = asari.fit_exponentials(q["t"][m], q["ratio"][m], 1, kappa=1.0, sign=-1)
            g2 = asari.fit_exponentials(q["t"][m], q["ratio_raw"][m], 1, kappa=1.0, sign=-1)
            fitq.append(dict(contexts=sil, over=reading, range=hi, Q_a1=f["params"][0][0], Q_tau1=f["params"][0][1],
                             ratio_1_plus_a1=1 + g["params"][0][0], ratio_tau1=g["params"][0][1],
                             raw_1_plus_a1=1 + g2["params"][0][0], raw_tau1=g2["params"][0][1]))
fitq = pd.DataFrame(fitq)
fitq["Q_ok"] = ((fitq.Q_a1 - 0.49).abs() <= 0.049) & ((fitq.Q_tau1 - 1.04).abs() <= 0.104)
print(fitq[["contexts", "over", "range", "Q_a1", "Q_tau1", "Q_ok"]].round(3).to_string())
q = qcurves[("silence counted", "running")]
print("\nQ(t) at selected times (silence counted, running):",
      {s: round(float(q["Q"][int(s * 1000)]), 3) for s in [0, 0.1, 0.25, 0.5, 1, 1.5, 2, 3, 4, 5, 6]})
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(q["t"], q["Q"], "k", lw=1)
r0 = fitq.iloc[0]
ax.plot(q["t"], r0.Q_a1 * np.exp(-q["t"] / r0.Q_tau1), color="0.6")
ax.set_xlim(0, 6.1); ax.set_ylim(-0.5, 1.5); ax.set_xlabel("time from probe onset (s)"); ax.set_ylabel("normalized power Q")
print("points outside the plotted y range [-0.5, 1.5]:", int(np.sum((q["Q"] < -0.5) | (q["Q"] > 1.5))),
      "; Q range", np.round([np.nanmin(q["Q"]), np.nanmax(q["Q"])], 3))

**Result: reproduced (resting on silence counted as a context).** With silence among the contexts, as in the authors' CxtDep.m and Figure 2B, Q(t) decays from 0.487 to 0.495 with time constants of 1.008 to 1.050 s in all four readings of the average and fit range (paper 0.49 and 1.04 s). Leaving silence out gives sizes of 0.553 to 0.555 and constants of 0.545 to 0.549 s, outside tolerance.

## timescale-longer-than-membrane

> The trial-to-trial noise autocorrelation and the autocorrelation of spontaneous activity decay
> within about 100 ms, much faster than the second-long timescale of context dependence in A1.

In [ ]:
lag_ms = np.arange(501)
nac = np.nanmean([p["noise_ac"] for p in el_nat], 0)
sac = np.nanmean([c["spont_ac"] for c in nat_cells if c.get("spont_ac") is not None], 0)
sac1 = np.nanmean([c["spont_ac_late"] for c in nat_cells if c.get("spont_ac_late") is not None], 0)
def first_below(a, v):
    return int(np.argmax(a < v)) if (a < v).any() else None
tab = pd.DataFrame([dict(series=n, lag_1e_ms=first_below(a, 1 / np.e), lag_0p1_ms=first_below(a, 0.1),
                         ac_at_100ms=a[100], ac_at_200ms=a[200])
                    for n, a in [("noise (Eq. 14, 45), probes", nac), ("spontaneous, whole interval", sac),
                                 ("spontaneous, without first 1 s", sac1)]])
print(tab.round(3).to_string())
print("probes:", len(el_nat), "| files with spontaneous segments:", sum(c.get("spont_ac") is not None for c in nat_cells))
print("one third of 1.04 s = 347 ms")

**Result: reproduced.** The noise autocorrelation falls below 1/e at 71 ms and below 0.1 at 115 ms; the autocorrelation of the intersequence silences falls below 1/e at 77 ms (75 ms without the first second) and below 0.1 at 121 ms (118 ms). All are under 150 ms, and the 1/e lags are below 347 ms, a third of 1.04 s.

## fig4-predictability-upper-bound

> Without knowledge of the context, no model can capture more than about half (0.51) of the
> predictable response power with a window of about 100 ms; the upper bound approaches one only
> with a window of seconds (time constant 1.04 s).

Eq. 43, Eq. 18 over the smoothed Eq. 17 (main) or the unsmoothed Eq. 17 (variant), one
exponential with constant 1; readings as for fig3b.

In [ ]:
fitq["ratio_ok"] = ((fitq.ratio_1_plus_a1 - 0.51).abs() <= 0.051) & ((fitq.ratio_tau1 - 1.04).abs() <= 0.104)
fitq["raw_ok"] = ((fitq.raw_1_plus_a1 - 0.51).abs() <= 0.051) & ((fitq.raw_tau1 - 1.04).abs() <= 0.104)
print(fitq[["contexts", "over", "range", "ratio_1_plus_a1", "ratio_tau1", "ratio_ok", "raw_1_plus_a1", "raw_tau1", "raw_ok"]].round(3).to_string())
q = qcurves[("silence counted", "running")]
print("\nratio (smoothed) at", {s: round(float(q["ratio"][int(s * 1000)]), 3) for s in [0, 0.1, 0.5, 1, 2, 3, 4]})
print("ratio (unsmoothed) at", {s: round(float(q["ratio_raw"][int(s * 1000)]), 3) for s in [0, 0.1, 0.5, 1, 2, 3, 4]})

**Result: partially reproduced.** The ratio of context-independent to stimulus-related power starts near half: with silence counted, 1 + alpha1 is 0.445 to 0.496 with the smoothed denominator and 0.495 to 0.502 unsmoothed (paper 0.51). Its approach to one is slower than the paper's in every reading: time constants of 1.70 to 2.33 s smoothed and 1.20 to 1.24 s unsmoothed (paper 1.04 s); the smoothed fits over all probes to 6 s run to the bound of 100 s. With silence left out the constants are 1.26 to 1.59 s smoothed and 0.70 s unsmoothed. No reading has both values within tolerance.

## fig4-strf-performance-low

> Linear STRF models of 20 A1 cells from Machens et al. (2004) predicted less than about 20% of
> the predictable response power, and extending the window from 180 ms to about 4 s did not
> improve them significantly, with or without a static nonlinearity; the LN model did only
> slightly better.

Not run here; see the record. The Wehr data (`wehr/`) are in the data set with the authors'
`STRFanalysis.m`.

In [ ]:
wehr_dirs = sorted({m.split("/")[2] for m in A.members if m.startswith("wehr/Results/") and m.count("/") >= 3})
print("Wehr recording folders:", len(wehr_dirs))

## fig5-example-conductances and fig6-conductance-decay

> In an example voltage-clamped A1 cell, excitatory and inhibitory conductances ... (fig5)
> Across 29 probes in 14 voltage-clamped cells, context dependence of both excitatory and
> inhibitory conductances decayed with a long constant of seconds ... (fig6)

Blocked: the data set holds no voltage-clamp recording (recording types in the data check).

In [ ]:
print("recording types of every file read:", dict(chk.groupby("type").size()))
print("Summary.mat experiment descriptions:", list(rows.experiment_description.unique()))

## fig8a-proportion-by-property

> When only one property of synthetic conditioning stimuli varied, context dependence was found
> in 77 of 95 probes (81.1%, 31 cells) for amplitude, 73 of 110 (35 cells) for frequency, 62 of 96
> (64.6%, 27 cells) for AM, 47 of 82 (57.3%, 25 cells) for FM and 59 of 137 (43.1%, 27 cells) for
> higher-order properties.

Each probe is tested with only the conditioning stimuli that differ in one property
(`asari.property_classes`, from the parameters stored with each fragment). A probe can have more
than one such ensemble (for example AM variants at each of four FM settings). Readings of what
Table 1 counts: `probe` (a probe counts once per property, significant when any of its ensembles
is) and `entry` (each probe x ensemble counts).

In [ ]:
se = pd.DataFrame([{k: e[k] for k in ["member", "cell", "probe", "probe_type", "property", "eligible", "n_trials"]}
                   | {"n_ctx": len(e["contexts"]), "significant": e.get("significant", False), "idx": i}
                   for i, e in enumerate(syn_entries)])
se["family"] = se.member.map(fam)
print("probe x ensemble entries:", len(se), "| eligible (>= 4 trials per context):", int(se.eligible.sum()),
      "| dropped:", dict(se[~se.eligible].groupby(["property", "n_trials"]).size()))
see = se[se.eligible]
print("\nTable 1 layout (probe type x property): probes (cells) | entries")
t1 = see.groupby(["probe_type", "property"]).agg(probes=("member", lambda s: len(set(zip(s, see.loc[s.index, "probe"])))),
                                                  cells=("cell", "nunique"), entries=("idx", "size"))
print(t1.to_string())
paper = {"amplitude": (77, 95), "frequency": (73, 110), "AM": (62, 96), "FM": (47, 82), "higher-order": (59, 137)}
rows8 = []
for prop, (ps, pn) in paper.items():
    s = see[see.property == prop]
    byp = s.groupby(["member", "probe"]).significant.any()
    rows8.append(dict(property=prop, paper_sig=ps, paper_n=pn, probe_n=len(byp), probe_sig=int(byp.sum()),
                      entry_n=len(s), entry_sig=int(s.significant.sum()), cells=s.cell.nunique()))
f8 = pd.DataFrame(rows8)
f8["probe_frac"] = f8.probe_sig / f8.probe_n
f8["entry_frac"] = f8.entry_sig / f8.entry_n
f8["paper_frac"] = f8.paper_sig / f8.paper_n
f8["reading_matching_n"] = np.where(f8.probe_n == f8.paper_n, "probe", np.where(f8.entry_n == f8.paper_n, "entry", "neither"))
f8["probe_ok"] = (f8.probe_sig - f8.paper_sig).abs() <= 0.1 * f8.paper_sig
f8["entry_ok"] = (f8.entry_sig - f8.paper_sig).abs() <= 0.1 * f8.paper_sig
print()
print(f8.round(3).to_string())

In [ ]:
# the reading whose denominator matches Table 1 (probe for amplitude, frequency, higher-order; entry
# for AM and FM), with the order checks of the partial rule
adopt = {"amplitude": "probe", "frequency": "probe", "AM": "entry", "FM": "entry", "higher-order": "probe"}
fr = {r.property: (r.probe_frac if adopt[r.property] == "probe" else r.entry_frac) for r in f8.itertuples()}
sg = {r.property: (r.probe_sig if adopt[r.property] == "probe" else r.entry_sig) for r in f8.itertuples()}
print("adopted reading, significant counts:", sg)
print("adopted reading, fractions:", {k: round(v, 3) for k, v in fr.items()})
print("amplitude > frequency:", fr["amplitude"] > fr["frequency"], "| AM > FM > higher-order:",
      fr["AM"] > fr["FM"] > fr["higher-order"])
for rd in ["probe", "entry"]:
    ff = dict(zip(f8.property, f8[rd + "_frac"]))
    print(f"{rd} reading: amplitude > frequency {ff['amplitude'] > ff['frequency']}, AM > FM > higher-order {ff['AM'] > ff['FM'] > ff['higher-order']}")

**Result: partially reproduced (resting on the grouping of conditioning stimuli).** The ensembles built from the stored stimulus parameters give Table 1's denominators exactly for amplitude (95) and frequency (110) when probes are counted, for AM (96) and FM (82) when probe x ensemble entries are counted, and 131 against 137 for higher-order properties. Significant counts are lower than the paper's: 58 of 95 for amplitude (paper 77), 62 of 110 for frequency (73), 41 of 96 for AM (62), 29 of 82 for FM (47), and 64 of 131 for higher-order (59, the only one within tolerance). Under both the probe and the entry reading the proportions keep the order amplitude > frequency and AM > FM > higher-order.

## fig8b-decay-by-property

> The normalized context-dependent power decays slowly for amplitude and frequency changes (time
> constants 0.66 and 0.97 s) and fast for AM, FM and higher-order changes (0.11, 0.15 and 0.20 s),
> so higher-order properties act mainly on a timescale of about 100 ms.

Q(t) per property over the probes of fig8a; one exponential with constant 0 from 0 to 4 s (main)
and from 0 to the shortest probe length of the property (variant). Readings of which entries form
a property's population: every entry, or one ensemble per probe (the one with the most
conditioning stimuli).

In [ ]:
def entries_for(prop, reading):
    s = see[see.property == prop]
    if reading == "probe":
        s = s.sort_values("n_ctx", ascending=False).drop_duplicates(["member", "probe"])
    return [syn_entries[i] for i in s.idx]

paper8b = {"amplitude": (0.57, 0.66), "frequency": (0.54, 0.97), "AM": (0.86, 0.11), "FM": (0.64, 0.15), "higher-order": (0.52, 0.20)}
rows8b, q8 = [], {}
for prop in paper8b:
    for reading in ["entry", "probe"]:
        ents = entries_for(prop, reading)
        q = asari.q_curves(ents, "running")
        q8[(prop, reading)] = q
        tmin = min(e["T"] for e in ents) / asari.SF
        for rng_name, hi in [("0-4 s", 4.0), ("0-shortest", tmin)]:
            m = (q["t"] <= hi) & np.isfinite(q["Q"])
            f = asari.fit_exponentials(q["t"][m], q["Q"][m], 1, kappa=0.0)
            rows8b.append(dict(property=prop, reading=reading, n=len(ents), range=rng_name, hi=round(hi, 2),
                               a1=f["params"][0][0], tau1=f["params"][0][1], paper_a1=paper8b[prop][0], paper_tau1=paper8b[prop][1]))
f8b = pd.DataFrame(rows8b)
f8b["tau_ok"] = (f8b.tau1 - f8b.paper_tau1).abs() <= 0.1 * f8b.paper_tau1
f8b["a_ok"] = (f8b.a1 - f8b.paper_a1).abs() <= 0.1 * f8b.paper_a1
print(f8b.round(3).to_string())
for (reading, rng_name), g in f8b.groupby(["reading", "range"]):
    tau = dict(zip(g.property, g.tau1))
    print(f"{reading}, {rng_name}: all tau within tolerance {bool(g.tau_ok.all())}; amplitude and frequency > 0.4 s "
          f"{tau['amplitude'] > 0.4 and tau['frequency'] > 0.4}; AM, FM, higher-order < 0.3 s "
          f"{all(tau[k] < 0.3 for k in ['AM', 'FM', 'higher-order'])}")

**Result: partially reproduced (resting on the grouping of conditioning stimuli).** The amplitude and frequency constants exceed 0.4 s (amplitude 1.35 s, frequency 0.78 to 0.84 s; paper 0.66 and 0.97 s) and the AM, FM and higher-order constants are below 0.3 s (AM 0.109 to 0.118 s, FM 0.113 to 0.278 s, higher-order 0.176 to 0.224 s; paper 0.11, 0.15 and 0.20 s) in every reading. Within tolerance are only the AM constant and size and, counting probes, the frequency size (0.497 against 0.54).

## fig8a-significance-fits

> The proportion of probes with significant context dependence decays fastest and is smallest
> for higher-order changes, with fitted (size, time constant) of (0.23, 0.25 s) for amplitude,
> (0.19, 0.27 s) for frequency, (0.13, 0.17 s) for AM, (0.07, 0.15 s) for FM and (0.04, 0.13 s)
> for higher-order properties.

One exponential, constant fixed at the property's mean noise floor (1000 shuffles per probe), 0
to 4 s; proportion over the probes still running at t (main) or over all; populations as in
fig8b. Size bounds: none, or non-negative.

In [ ]:
paper8a = {"amplitude": (0.23, 0.25, 2.8e-3), "frequency": (0.19, 0.27, 2.3e-3), "AM": (0.13, 0.17, 2.3e-3),
           "FM": (0.07, 0.15, 2.1e-3), "higher-order": (0.04, 0.13, 3.6e-3)}
rows8f = []
for prop in paper8a:
    for reading in ["entry", "probe"]:
        ents = entries_for(prop, reading)
        for over in ["running", "all"]:
            prop_c = asari.population_mean(asari.stack(ents, "signif"), over)
            noise_c = asari.population_mean(asari.stack(ents, "noise"), over)
            tt = np.arange(len(prop_c)) / 1000
            m = tt <= 4.0
            kap = float(np.nanmean(noise_c[:4000]))
            for nonneg in [False, True]:
                f = asari.fit_exponentials(tt[m], prop_c[m], 1, kappa=kap, nonneg=nonneg)
                rows8f.append(dict(property=prop, reading=reading, over=over, nonneg=nonneg, n=len(ents),
                                   a1=f["params"][0][0], tau1=f["params"][0][1], kappa=kap,
                                   paper_a1=paper8a[prop][0], paper_tau1=paper8a[prop][1], paper_kappa=paper8a[prop][2]))
f8f = pd.DataFrame(rows8f)
f8f["ok"] = ((f8f.a1 - f8f.paper_a1).abs() <= 0.1 * f8f.paper_a1) & ((f8f.tau1 - f8f.paper_tau1).abs() <= 0.1 * f8f.paper_tau1)
print(f8f.round(4).to_string())
order = ["amplitude", "frequency", "AM", "FM", "higher-order"]
for (reading, over, nonneg), g in f8f.groupby(["reading", "over", "nonneg"]):
    a = dict(zip(g.property, g.a1))
    print(f"{reading}/{over}/nonneg={nonneg}: all ten within tolerance {bool(g.ok.all())}; values within tolerance "
          f"{int(g.ok.sum())} of 5 properties; sizes in the printed order {all(a[order[i]] > a[order[i + 1]] for i in range(4))}")

**Result: not reproduced.** No property has its size and time constant both within tolerance in any reading. The sizes are smaller than the paper's (amplitude 0.119 to 0.125 against 0.23, frequency 0.147 against 0.19, AM 0.064 to 0.073 against 0.13, FM 0.012 to 0.036 against 0.07, higher-order 0.026 to 0.036 against 0.04) and their order fails at the first step, amplitude below frequency. The time constants are close to the printed ones (amplitude 0.27 s, frequency 0.28 to 0.29 s, AM 0.20 s, FM 0.13 s, higher-order 0.11 to 0.13 s), and the fitted noise floors are 0.0020 to 0.0033 (paper 0.0021 to 0.0036).

## fig8c-relative-effects

> Changes in amplitude and frequency of the conditioning stimuli produced context effects as large
> and long-lasting as natural sound contexts, while AM, FM and higher-order changes produced
> substantially smaller and shorter effects; A1 is more sensitive to lower-order than to
> higher-order properties.

Area under Q(t) from 0 to 4 s for each property over the area for natural sounds (Figure 3B,
silence counted, probes running at t). Confidence intervals: 200 resamplings of 1000 probes drawn
with replacement (the probe reading of "sample"); the natural-sound denominator is resampled in
the same way. The trial reading of "sample" was not run (see the record).

In [ ]:
rng8c = np.random.default_rng(80_003)

def q_area(ents, idx=None):
    if idx is not None:
        ents = [ents[i] for i in idx]
    q = asari.q_curves(ents, "running")
    m = q["t"] < 4.0
    return float(np.nansum(q["Q"][m]) * 1e-3)

nat_area = q_area(el_nat)
rows8c = []
for reading in ["entry", "probe"]:
    for prop in order:
        ents = entries_for(prop, reading)
        point = q_area(ents) / nat_area
        boots = []
        for b in range(200):
            i1 = rng8c.integers(0, len(ents), 1000)
            i2 = rng8c.integers(0, len(el_nat), 1000)
            boots.append(q_area(ents, i1) / q_area(el_nat, i2))
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rows8c.append(dict(reading=reading, property=prop, n=len(ents), area=q_area(ents), ratio=point, ci_lo=lo, ci_hi=hi))
f8c = pd.DataFrame(rows8c)
print("natural-sound area 0-4 s:", round(nat_area, 3), "s")
print(f8c.round(3).to_string())
for reading, g in f8c.groupby("reading"):
    d = g.set_index("property")
    low_ok = all(d.ci_hi[k] >= 1 for k in ["amplitude", "frequency"])
    high_lo = min(d.ci_lo["amplitude"], d.ci_lo["frequency"])
    hi_ok = all(d.ci_hi[k] < 1 and d.ci_hi[k] < high_lo for k in ["AM", "FM", "higher-order"])
    pt = all(d.ratio[k] >= 1 for k in ["amplitude", "frequency"]) and all(
        d.ratio[k] < 1 and d.ratio[k] < min(d.ratio["amplitude"], d.ratio["frequency"]) for k in ["AM", "FM", "higher-order"])
    print(f"{reading}: amplitude and frequency intervals include or exceed 1: {low_ok}; AM, FM, higher-order intervals "
          f"below 1 and below the amplitude/frequency lower bounds: {hi_ok}; point estimates in that pattern: {pt}")

**Result: partially reproduced (resting on the grouping of conditioning stimuli).** Relative to natural sounds, the area under Q from 0 to 4 s is 0.97 [0.85, 1.13] for amplitude and 0.92 [0.81, 1.04] for frequency, so both intervals include 1, and 0.21 [0.16, 0.26] for AM, 0.19 [0.14, 0.26] for FM and 0.54 [0.46, 0.64] for higher-order properties, all below 1 and below the amplitude and frequency lower bounds (entry reading). Counting each probe once, the higher-order interval [0.64, 0.83] overlaps the frequency lower bound of 0.79, and the criterion fails on that comparison alone. The trial resampling reading was not run.

## fig8-natural-synthetic-probes-alike

> Natural and synthetic probes gave no difference in the context effects of the synthetic
> conditioning stimuli, so they were combined.

Fisher's exact test per property on the proportion of significant probes, natural against
synthetic (TORC, DMR, MHT, MCN) probes, Bonferroni over the properties tested with both
(criterion). Shown for the probe and the entry readings of fig8a.

In [ ]:
rows8n = []
for reading in ["probe", "entry"]:
    for prop in order:
        s = see[see.property == prop].copy()
        if reading == "probe":
            s = s.groupby(["member", "probe"]).agg(significant=("significant", "any"), probe_type=("probe_type", "first")).reset_index()
        s["natural"] = s.probe_type == "natural"
        a = int((s.natural & s.significant).sum()); b = int((s.natural & ~s.significant).sum())
        c = int((~s.natural & s.significant).sum()); d = int((~s.natural & ~s.significant).sum())
        if (a + b) and (c + d):
            orr, p = stats.fisher_exact([[a, b], [c, d]])
        else:
            orr, p = np.nan, np.nan
        rows8n.append(dict(reading=reading, property=prop, natural=f"{a}/{a + b}", synthetic=f"{c}/{c + d}",
                           nat_frac=a / (a + b) if a + b else np.nan, syn_frac=c / (c + d) if c + d else np.nan,
                           odds_ratio=orr, p=p))
f8n = pd.DataFrame(rows8n)
for reading, g in f8n.groupby("reading"):
    k = int(g.p.notna().sum())
    f8n.loc[g.index, "tested"] = k
    f8n.loc[g.index, "p_bonferroni"] = np.minimum(g.p * k, 1)
print(f8n.round(4).to_string())

**Result: reproduced (the test is this file's, not the paper's).** No property shows a difference between natural and synthetic probes: Fisher's exact test gives Bonferroni-corrected p of 1.0 for amplitude (27/45 against 31/50), frequency (23/46 against 39/64), AM (17/25 against 8/14) and higher-order properties (29/58 against 35/73), and 0.80 or more in the entry reading. FM has no synthetic probes and is not tested.

## fig9-mgb-fast-decay

> In the auditory thalamus (MGB), context dependence of spike responses decayed within a short time
> (time constant about 80 ms; 93 probes in 14 cells), so the long-lasting component in A1 arises
> mainly in cortex.

PSTHs in 100-ms bins (k = 0 to 60), SD over contexts (n - 1), averaged over probes; one exponential
with a free constant. Readings: time of a bin at its start or its centre; average over probes
still running or over all (ended probes as 0); spike detectors as in a1-mgb-firing-rates.

In [ ]:
rows9 = []
for det, (infos, probes) in mgb.items():
    el = [p for p in probes if p["eligible"]]
    M = np.full((len(el), 61), np.nan)
    for k, p in enumerate(el):
        n = min(61, len(p["sd"]))
        M[k, :n] = p["sd"][:n]
    for over in ["running", "all"]:
        msd = asari.population_mean(M, over)
        for tb in ["start", "centre"]:
            tt = np.arange(61) * 0.1 + (0.05 if tb == "centre" else 0.0)
            f = asari.fit_exponentials(tt, msd, 1, kappa=None)
            rows9.append(dict(detector=det, over=over, bin_time=tb, probes=len(el), cells=len({p["cell"] for p in el}),
                              a1=f["params"][0][0], tau1=f["params"][0][1], kappa=f["kappa"],
                              mean_sd_after_0p5s=float(np.nanmean(msd[6:])), first_bins=np.round(msd[:4], 2).tolist()))
f9 = pd.DataFrame(rows9)
f9["tau_ok"] = (f9.tau1 - 0.08).abs() <= 0.008
f9["tau_below_0p1"] = f9.tau1 < 0.1
f9["plateau_vs_fit_kappa"] = (f9.mean_sd_after_0p5s - f9.kappa).abs() <= 0.25 * f9.kappa.abs()
f9["plateau_vs_paper_kappa"] = (f9.mean_sd_after_0p5s - 2.0).abs() <= 0.5
print(f9.round(3).to_string())

**Result: reproduced.** The mean SD of PSTHs over contexts in 93 probes of 14 MGB cells decays with a time constant of 0.060 s (probes running) or 0.079 to 0.080 s (all probes), below 0.1 s in every reading and within tolerance of 0.080 s over all probes, and stays at the fitted constant after 0.5 s (1.13 against 1.14 spikes/s; 0.77 against 0.79). The size (5.3 to 12.2 spikes/s against 15.0) and constant (0.79 to 1.14 against 2.0) are lower than the paper's with the authors' detector; with robust thresholds the time constants are 0.063 to 0.090 s.

## fig9-example-first-bin

> In an example MGB unit, the response to a probe differed between contexts only in the first
> 100-ms bin of the PSTH.

In [ ]:
for det, (infos, probes) in mgb.items():
    el = [p for p in probes if p["eligible"]]
    fb = [bool(p["sd"][0] >= 2 * np.max(p["sd"][1:])) for p in el]
    design = [p for p in el if p["n_contexts"] == 3 and 0 in p["contexts"] and p["n_trials"] >= 10]
    fbd = [p for p in design if p["sd"][0] >= 2 * np.max(p["sd"][1:])]
    print(f"{det}: first-bin SD at least twice every later bin in {sum(fb)} of {len(el)} probes "
          f"({sum(fb) / len(el):.3f}); probes with silence and two fragments as contexts: {len(design)}, of which {len(fbd)} pass")
    if det == "getspikes":
        for p in design:
            print("   ", p["cell"], "probe", p["probe"], "trials", p["n_trials"], "contexts", p["contexts"],
                  "SD first bins", np.round(p["sd"][:4], 2), "max later", round(float(np.max(p["sd"][1:])), 2))

**Result: reproduced.** Of the 12 MGB probes heard after silence and two fragments with at least 10 trials, 4 (all in 20071010-ha2-003) have an SD over contexts in the first 100-ms bin at least twice that of every later bin (first bin 26 to 31 spikes/s against at most 11 to 15 later). Over all 93 probes this holds for 38 (0.41).

## fig7-example-frequency-context

> In an example A1 neuron, a natural sound probe preceded by three DMR contexts that differed only
> in frequency band gave significant context dependence for about 4 s.

Blocked: no recording has DMR conditioning stimuli in the bands 0.625-2.5, 2.5-10 and 10-40 kHz.
The search over the stored bands of every synthetic recording:

In [ ]:
target = {(625.0, 2500.0), (2500.0, 10000.0), (10000.0, 40000.0)}
for i in syn_infos:
    b = sorted({x for x in i["bands"] if x is not None})
    if b:
        print(i["cell"], i["trigger_type"], "bands (Hz):", b, "| has all three Figure 7 bands:", target <= set(b))
nb_torc = [i["cell"] for i in syn_infos if any("bandwidth variants" in d for d in i["descriptions"]) and not any(i["bands"])]
print("TORC bandwidth-variant recordings without stored bands:", nb_torc)

## figs1-silence-probe

> A subset of cells showed significant context dependence soon after the onset of the
> intersequence silences, but the response power attributable to the preceding stimulus alone
> faded within about 1 s after the stimulus ended.

The silence after each sequence as a probe (2.5 s, cut to the shortest gap; CxtDep.m), its
context the last fragment of the sequence; significance as for the other probes; power Eq. 17
(main) and Eq. 19 (variant) averaged over the silence probes.

In [ ]:
sp = [(c["cell"], c["silence_probe"]) for c in nat_cells if c.get("silence_probe") is not None]
print("natural-sound files with a silence probe (>= 2 contexts, >= 4 trials):", len(sp), "of", len(nat_cells),
      "| cells:", len({c for c, _ in sp}))
early = [(c, s["runs"][0]) for c, s in sp if s["runs"] and s["runs"][0][0] < 0.5]
print("silence probes with a significant period starting within 0.5 s:", len(early), "| cells:", len({c for c, _ in early}))
print("  first periods:", [(c, (round(a, 3), round(b, 3))) for c, (a, b) in early][:12])
pred = asari.population_mean(asari.stack([s for _, s in sp], "predictable"), "running")
inv = asari.population_mean(asari.stack([s for _, s in sp], "invariant"), "running")
dep = pred - inv
tt = np.arange(len(pred)) / 1000
for name, y in [("Eq. 17 (stimulus-related)", pred), ("Eq. 19 (context-dependent)", dep)]:
    y0 = float(np.mean(y[:10]))
    below = np.nonzero(y < 0.1 * y0)[0]
    print(f"{name}: mean over first 10 ms {y0:.3f} mV^2; first time below 10% of it: "
          f"{below[0] / 1000 if len(below) else None} s; at 0.5, 1.0, 1.1, 2.0 s:",
          [round(float(y[int(s * 1000)]), 3) for s in [0.5, 1.0, 1.1, 2.0] if int(s * 1000) < len(y)])

**Result: reproduced.** Treating the intersequence silence as a probe, 22 silence probes in 19 cells have a significant period starting within 0.5 s of silence onset, most within the first 0.1 s. The stimulus-related power (Eq. 17) over the 44 silence probes falls from 5.59 mV^2 to below 10% of it at 0.541 s, before 1.1 s; the context-dependent power (Eq. 19) does so at 0.458 s.

## Summary

| claim | outcome |
|---|---|
| methods-recording-counts | partially reproduced |
| a1-mgb-firing-rates | not reproduced (assumption: spike detectors) |
| fig2-example-reliability | reproduced |
| fig2-example-long-context | reproduced |
| fig3a-proportion-significant | partially reproduced |
| fig3a-significance-decay | partially reproduced (assumption: non-negative sizes) |
| fig3b-fractional-power-decay | reproduced (assumption: silence as a context) |
| timescale-longer-than-membrane | reproduced |
| fig4-predictability-upper-bound | partially reproduced |
| fig4-strf-performance-low | blocked: not run in this attempt |
| fig5-example-conductances | blocked: data missing |
| fig6-conductance-decay | blocked: data missing |
| fig8a-proportion-by-property | partially reproduced (assumption: ensembles) |
| fig8b-decay-by-property | partially reproduced (assumption: ensembles) |
| fig8a-significance-fits | not reproduced |
| fig8c-relative-effects | partially reproduced (assumption: ensembles) |
| fig8-natural-synthetic-probes-alike | reproduced (assumption: the test) |
| fig9-mgb-fast-decay | reproduced |
| fig9-example-first-bin | reproduced |
| fig7-example-frequency-context | blocked: data missing |
| figs1-silence-probe | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method: details the Methods leave open are taken from the authors' scripts shipped with the data (CxtDep.m and its helpers), and the readings the claims file names are all shown above. A1 spikes are detected by a fixed threshold, not by hand. The ensembles of conditioning stimuli that differ in one property are this attempt's reading of the stored stimulus parameters. Exponential fits use scipy's least squares with a grid of start values.

What was found about the data set: it holds the natural-sound A1 data (39 cells, 305 probes), the MGB data (14 cells, 93 probes) and the synthetic-sound A1 data of the paper, with the authors' analysis scripts, but no voltage-clamp recording. In two cells (20051108-ha1-001 in A1 and 20071010-ha2-005 in MGB) the fragment lists stored with the cell's two files are swapped; each file's stored durations fit the other file's triggers exactly. In 20061201-ha2-001 every sequence trigger is 308 ms longer than its seven stored fragment durations. One synthetic file (20061231-ha2-001) has a sequence presented 3 times against 4 stated effective repeats.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")